# Example 13: Synthetic families and coupled fringes

Audience:
- Mathematicians using the synthetic-data helpers to build small, interpretable multiparameter examples.

Prerequisites:
- Basic familiarity with `SyntheticFamily`, finite encodings, and the visualization workflow from examples 10-12.
- A working `TamerOp` environment.

Learning goals:
- Build parameter sweeps from the coupled synthetic generators.
- Inspect family members before doing any heavier encoding work.
- Compare a diagonal box fringe and a coupled staircase box fringe through the canonical visualization/export workflow.


## Outline

1. Setup and constants
2. Build sweep families from the coupled synthetic generators
3. Compare one diagonal box fringe and one coupled staircase box fringe
4. Build inspectable visual specs
5. Save the final visuals with the canonical batch export helper


In [1]:
include(joinpath(@__DIR__, "_common.jl"))

outdir = example_outdir("13_synthetic_families_and_coupled_fringes")

const SD = TO.SyntheticData
const FZ = TO.FlangeZn
const TOA = TO.Advanced

nothing


## 1. Build sweep families from the coupled synthetic generators

The `sweep_family(...)` helper is the cheap-first way to inspect a parameter family before any encoding or invariant work. We will build three small families:

- a coupled chain family,
- a staircase-box family,
- and a mixed-face flange family.


In [2]:
chain_bars_a = [(1, 4), (2, 5), (3, 6)]
chain_bars_b = [(1, 5), (3, 6), (4, 7)]
chain_bars_c = [(1, 6), (2, 6), (3, 7)]

chain_family = SD.sweep_family(
    SD.coupled_chain_fringe;
    sweep=(bars=[chain_bars_a, chain_bars_b, chain_bars_c],),
    n=7,
    generator=:coupled_chain_sweep,
)

stairs_a = [([0.0, 0.0], [1.5, 1.0]), ([0.75, 0.5], [2.25, 1.75]), ([1.5, 1.0], [3.0, 2.5])]
stairs_b = [([0.0, 0.0], [2.0, 1.25]), ([0.75, 0.5], [2.75, 2.25]), ([1.5, 1.0], [3.75, 3.25])]

box_family = SD.sweep_family(
    SD.staircase_box_fringe;
    sweep=(bars=[stairs_a, stairs_b],),
    generator=:staircase_box_sweep,
)

phi_default = [1 1 0; 1 0 1; 0 1 1]
phi_sparse = [1 1 0; 0 1 1; 0 0 1]

flange_family = SD.sweep_family(
    SD.mixed_face_flange;
    sweep=(phi=[phi_default, phi_sparse],),
    generator=:mixed_face_flange_sweep,
)

(; chain=SD.synthetic_family_summary(chain_family),
   staircase=SD.synthetic_family_summary(box_family),
   flange=SD.synthetic_family_summary(flange_family))


(chain = (kind = :synthetic_family, generator = :coupled_chain_sweep, nitems = 3, item_kind = :fringe_module, parameter_keys = [:n, :bars], labels = ["n=7, bars=[(1, 4), (2, 5), (3, 6)]", "n=7, bars=[(1, 5), (3, 6), (4, 7)]", "n=7, bars=[(1, 6), (2, 6), (3, 7)]"]), staircase = (kind = :synthetic_family, generator = :staircase_box_sweep, nitems = 2, item_kind = :synthetic_box_fringe, parameter_keys = [:bars], labels = ["bars=[([0.0, 0.0], [1.5, 1.0]), ([0.75, 0.5], [2.25, 1.75]), ([1.5, 1.0], [3.0, 2.5])]", "bars=[([0.0, 0.0], [2.0, 1.25]), ([0.75, 0.5], [2.75, 2.25]), ([1.5, 1.0], [3.75, 3.25])]"]), flange = (kind = :synthetic_family, generator = :mixed_face_flange_sweep, nitems = 2, item_kind = :flange, parameter_keys = [:phi], labels = ["phi=[1 1 0; 1 0 1; 0 1 1]", "phi=[1 1 0; 0 1 1; 0 0 1]"]))

Inspect the family members directly before doing any encoding work. The point here is to confirm that the sweep parameters are producing the intended geometric/algebraic changes.


In [3]:
chain_items = [(label, TO.describe(item)) for (label, item) in zip(SD.synthetic_labels(chain_family), SD.synthetic_items(chain_family))]
box_items = [(label, SD.box_fringe_summary(item)) for (label, item) in zip(SD.synthetic_labels(box_family), SD.synthetic_items(box_family))]
flange_items = [(label, (; summary=TO.describe(item), dim_at_2_2=FZ.dim_at(item, (2, 2))))
                for (label, item) in zip(SD.synthetic_labels(flange_family), SD.synthetic_items(flange_family))]

(; chain_items, box_items, flange_items)


(chain_items = [("n=7, bars=[(1, 4), (2, 5), (3, 6)]", (kind = :fringe_module, field = Main.TamerOp.CoreModules.CoeffFields.QQField(), poset_kind = :finite_poset, nvertices = 7, ngenerators = 3, nrelations = 3, matrix_size = (3, 3), phi_density = 0.5555555555555556)), ("n=7, bars=[(1, 5), (3, 6), (4, 7)]", (kind = :fringe_module, field = Main.TamerOp.CoreModules.CoeffFields.QQField(), poset_kind = :finite_poset, nvertices = 7, ngenerators = 3, nrelations = 3, matrix_size = (3, 3), phi_density = 0.5555555555555556)), ("n=7, bars=[(1, 6), (2, 6), (3, 7)]", (kind = :fringe_module, field = Main.TamerOp.CoreModules.CoeffFields.QQField(), poset_kind = :finite_poset, nvertices = 7, ngenerators = 3, nrelations = 3, matrix_size = (3, 3), phi_density = 0.5555555555555556))], box_items = [("bars=[([0.0, 0.0], [1.5, 1.0]), ([0.75, 0.5], [2.25, 1.75]), ([1.5, 1.0], [3.0, 2.5])]", (kind = :synthetic_box_fringe, ambient_dim = 2, nupsets = 3, ndownsets = 3, matrix_size = (3, 3), field = Main.TamerOp.C

## 2. Compare one diagonal box fringe and one coupled staircase box fringe

Now pick one simple diagonal box pattern and one coupled staircase pattern. We encode both, build their rank invariants, and keep the cheap summaries visible at each step.


In [4]:
diagonal_boxes = [
    ([0.0, 0.0], [1.0, 1.0]),
    ([1.4, 0.2], [2.4, 1.2]),
    ([2.7, 0.8], [3.7, 1.8]),
]

box_diag = SD.box_bar_fringe(bars=diagonal_boxes)
box_coupled = SD.staircase_box_fringe(bars=stairs_b)

(; diagonal=SD.box_fringe_summary(box_diag),
   coupled=SD.box_fringe_summary(box_coupled))


(diagonal = (kind = :synthetic_box_fringe, ambient_dim = 2, nupsets = 3, ndownsets = 3, matrix_size = (3, 3), field = Main.TamerOp.CoreModules.CoeffFields.QQField()), coupled = (kind = :synthetic_box_fringe, ambient_dim = 2, nupsets = 3, ndownsets = 3, matrix_size = (3, 3), field = Main.TamerOp.CoreModules.CoeffFields.QQField()))

Because `SyntheticBoxFringe` now has a root workflow encoding path, this part of the notebook can stay on the simple surface. The synthetic provenance does not matter: `TO.encode(box_diag)` and `TO.encode(box_coupled)` return ordinary `EncodingResult`s, so the root workflow invariant wrappers work as well.


In [5]:
enc_diag = TO.encode(box_diag)
enc_coupled = TO.encode(box_coupled)

pi_diag = TO.encoding_map(enc_diag)
pi_coupled = TO.encoding_map(enc_coupled)
rank_diag = TO.rank_invariant(enc_diag)
rank_coupled = TO.rank_invariant(enc_coupled)

(; diagonal_encoding=TO.describe(enc_diag),
   coupled_encoding=TO.describe(enc_coupled),
   diagonal_rank=TO.describe(rank_diag),
   coupled_rank=TO.describe(rank_coupled))


(diagonal_encoding = (kind = :encoding_result, poset_type = Main.TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = Main.TamerOp.Modules.PModule{Rational{BigInt}, Main.TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, Main.TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_map_type = Main.TamerOp.EncodingCore.CompiledEncoding{Main.TamerOp.PLBackend.PLEncodingMapBoxes{2, 1, 1}, Main.TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Float64}, Vector{Float64}}, Vector{Tuple{Float64, Float64}}, Main.TamerOp.CoreModules.EncodingCache}, compiled = true, backend = :pl_backend, has_cohomology = true, has_presentation = true, module_dims = [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0]), coupled_encoding = (kind = :encoding_result, poset_type = Main.TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = Main.TamerOp.Modules.PModule{Rational{BigInt}, Main.TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, Main.TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_m

## 3. Build inspectable visual specs

Stay cheap-first here as well. Build inspectable `VisualizationSpec` objects, inspect their summaries, and only then decide whether to render or save the actual visuals.


In [6]:
regions_diag_spec = TOA.visual_spec(pi_diag; kind=:regions)
regions_coupled_spec = TOA.visual_spec(pi_coupled; kind=:regions)
rank_diag_spec = TOA.visual_spec(rank_diag; kind=:rank_heatmap)
rank_coupled_spec = TOA.visual_spec(rank_coupled; kind=:rank_heatmap)

(; diagonal_regions=TOA.visual_summary(regions_diag_spec),
   coupled_regions=TOA.visual_summary(regions_coupled_spec),
   diagonal_rank=TOA.visual_summary(rank_diag_spec),
   coupled_rank=TOA.visual_summary(rank_coupled_spec))


(diagonal_regions = (kind = :visualization_spec, visual_kind = :regions, title = "Box encoding", subtitle = "", nlayers = 27, npanels = 0, layer_types = (:RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :PointLayer), axes = (xlabel = "x1", ylabel = "x2", xlimits = (-2.0, 5.7), ylimits = (-2.0, 1.9), zlabel = "z", zlimits = nothing, aspect = :data, xticks = nothing, yticks = nothing), metadata = (object = :pl_boxes, nregions = 13, ncells = 49, query_count = 0, figure_size = (860, 620), legend_position = :right), legend_visible = true, interaction = (hover = true, labels = false, clicks = false, widgets = (), notebook = :summary_card)), coupled_regions = (kind = :visualization_spec, visual_k

If you want to render one visual inline before saving, the canonical call is still `TO.visualize(...)`.


In [7]:
TO.available_visuals(rank_diag), TO.available_visuals(rank_coupled)


((:rank_heatmap, :rank_rectangles), (:rank_heatmap, :rank_rectangles))

## 4. Save the final visuals

Use the simple batch export helper. The library will choose PNG when static export is available and fall back to HTML automatically.


In [8]:
exports = TO.save_visuals(outdir,
                          [
                              (; stem="diagonal_box_regions", obj=pi_diag, kind=:regions),
                              (; stem="coupled_box_regions", obj=pi_coupled, kind=:regions),
                              (; stem="diagonal_box_rank_heatmap", obj=rank_diag, kind=:rank_heatmap),
                              (; stem="coupled_box_rank_heatmap", obj=rank_coupled, kind=:rank_heatmap),
                          ];
                          prefer=:static)

Dict(TO.export_stem(r) => TO.export_path(r) for r in exports)


Dict{String, String} with 4 entries:
  "diagonal_box_rank_heatmap" => "/home/eriknovak/Documents/duke_fall_2025/tame…
  "diagonal_box_regions"      => "/home/eriknovak/Documents/duke_fall_2025/tame…
  "coupled_box_rank_heatmap"  => "/home/eriknovak/Documents/duke_fall_2025/tame…
  "coupled_box_regions"       => "/home/eriknovak/Documents/duke_fall_2025/tame…

## Notes

- `sweep_family(...)` is the canonical entrypoint for synthetic parameter sweeps.
- The family summaries are the cheap-first inspection layer; they should come before encoding work.
- `SyntheticBoxFringe` now flows through the root workflow `encode(...)` surface, so synthetic provenance does not force a separate advanced encoding path.
- `TO.Advanced` is still used here for `visual_spec(...)`, which remains an advanced inspection surface.
- `save_visuals(...)` is the canonical export path for this notebook. It replaces manual backend probing and target bookkeeping.
